In [1]:
from pypot.dynamixel import DxlIO
from pypot.dynamixel.protocol.v1 import *

from glob import glob

ports = glob('/dev/ttyACM*')
assert len(ports) == 1

port = ports[0]
print('Connecting on port: {}'.format(port))
dxl_io = DxlIO(port)

N_AXIS = 2

Connecting on port: /dev/ttyACM0


In [2]:
def ping(poulpe_id):
    ping_packet = DxlPingPacket(poulpe_id)
    dxl_io._send_packet(ping_packet)

ping(42)

In [3]:
import struct

def read_current_pos(poulpe_id):
    pos_packet = DxlReadDataPacket(poulpe_id, 50, N_AXIS * 4)
    resp = dxl_io._send_packet(pos_packet)
    data = bytearray(resp.parameters)
    pos = struct.unpack(N_AXIS * 'f', data)
    return pos

read_current_pos(42)

(0.0, 0.0)

In [4]:
import struct

def read_target_position(poulpe_id):
    pos_packet = DxlReadDataPacket(poulpe_id, 60, N_AXIS * 4)
    resp = dxl_io._send_packet(pos_packet)
    data = bytearray(resp.parameters)
    pos = struct.unpack(N_AXIS * 'f', data)
    return pos

read_target_position(42)

(0.10433026403188705, 0.1527533382177353)

In [5]:
def write_target_position(poulpe_id, target):
    p = DxlWriteDataPacket(poulpe_id, 60, struct.pack(N_AXIS * 'f', *target))
    resp = dxl_io._send_packet(p)
    return resp

write_target_position(42, [20.15, 0.5])

DxlStatusPacket(id=42, error=0, parameters=())

In [6]:
def read_torque_enabled(poulpe_id):
    p = DxlReadDataPacket(poulpe_id, 40, N_AXIS)
    resp = dxl_io._send_packet(p)
    data = bytearray(resp.parameters)
    torque = struct.unpack(N_AXIS * '?', data)
    return torque

read_torque_enabled(42)

(False, False)

In [7]:
def write_torque_enabled(poulpe_id, torque):
    p = DxlWriteDataPacket(poulpe_id, 40, struct.pack(N_AXIS * '?', *torque))
    resp = dxl_io._send_packet(p)
    return resp

write_torque_enabled(42, [False, False])

DxlStatusPacket(id=42, error=0, parameters=())

In [8]:
import time
import numpy as np

pos = []
send_target = []
read_target = []

id = 42

t0 = time.time()
while True:
    if time.time() - t0 > 5:
        break

    target = [
        np.random.rand(), 
        30 * np.sin(2 * np.pi * 0.5 * time.time()),
    ]
    write_target_position(id, target)
    send_target.append(target)

    pos.append(read_current_pos(id))
    read_target.append(read_target_position(id))


In [9]:
np.array(send_target) - np.array(read_target)

array([[-8.19754153e-10, -5.48291279e-07],
       [-1.21527749e-08,  7.42617392e-07],
       [-1.25288906e-08,  2.32809988e-07],
       ...,
       [ 7.97559874e-09,  7.09437437e-07],
       [ 3.58099439e-10, -1.14005719e-07],
       [-1.29756296e-08, -7.06303737e-07]])

In [10]:
def change_id(old_id, new_id):
    p = DxlWriteDataPacket(old_id, 3, struct.pack('B', new_id))
    resp = dxl_io._send_packet(p)
    return resp

change_id(42, 43)

DxlStatusPacket(id=42, error=0, parameters=())

In [11]:
ping(43)

In [12]:
read_current_pos(43)

(0.0, 0.0)

In [13]:
change_id(43, 42)

DxlStatusPacket(id=42, error=0, parameters=())

In [14]:
ping(42)

In [15]:
read_current_pos(42)

(0.0, 0.0)